# 📊 Performance_Analytics.ipynb
## Bluestock Fintech — Day 3 EDA | Fund Performance Analytics
> **40 Mutual Fund Schemes | Tasks 1–8 | Deliverables: fund_scorecard.csv · alpha_beta.csv · benchmark_comparison.png**

| Task | Metric | Formula |
|------|--------|---------|
| 1 | Daily Returns | `NAV_t / NAV_t-1 − 1` |
| 2 | CAGR 1/3/5yr | `(NAV_end/NAV_start)^(1/n) − 1` |
| 3 | Sharpe Ratio | `(Rp − Rf) / σp × √252`, Rf = 6.5% |
| 4 | Sortino Ratio | Same but σ = downside std only |
| 5 | Alpha & Beta | OLS on Nifty 100; α = intercept × 252 |
| 6 | Max Drawdown | `min(NAV / cummax − 1)` |
| 7 | Fund Score | 30% CAGR + 25% Sharpe + 20% Alpha + 15% Expense + 10% Drawdown |
| 8 | Benchmark | Top 5 vs Nifty 50/100; Tracking Error = std(fund−bm)×√252 |


## 0 · Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import linregress, skew, kurtosis
import warnings, os
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

OUTPUT = 'outputs'
os.makedirs(OUTPUT, exist_ok=True)
print('✓ All libraries imported')


## 1 · Data Generation (40 Schemes, 5-Year NAV History)
> **Note:** This notebook generates realistic synthetic data for 40 Indian mutual fund schemes.
> Replace the `nav_df` / `scheme_df` assignments below with `pd.read_sql_query(...)` calls when you have the live SQLite DB.

In [ ]:
np.random.seed(2024)

FUNDS = [
    # (name,  annual_return, volatility, expense_ratio)
    ("HDFC Mid-Cap Opportunities",       0.175, 0.18, 1.62),
    ("Axis Bluechip Fund",               0.138, 0.14, 1.43),
    ("Mirae Asset Large Cap",            0.142, 0.13, 0.52),
    ("Parag Parikh Flexi Cap",           0.162, 0.15, 0.63),
    ("Kotak Emerging Equity",            0.188, 0.20, 0.45),
    ("SBI Small Cap Fund",               0.202, 0.22, 1.75),
    ("Nippon India Small Cap",           0.195, 0.21, 1.83),
    ("ICICI Pru Bluechip",               0.133, 0.13, 1.15),
    ("UTI Flexi Cap Fund",               0.148, 0.15, 1.25),
    ("DSP Mid Cap Fund",                 0.169, 0.17, 1.85),
    ("Franklin India Prima Plus",        0.121, 0.14, 1.80),
    ("Canara Robeco Equity Hybrid",      0.118, 0.12, 1.70),
    ("Tata Large Cap Fund",              0.128, 0.13, 1.91),
    ("Aditya Birla SL Frontline Eq",     0.130, 0.14, 1.73),
    ("HDFC Flexicap Fund",               0.155, 0.16, 1.44),
    ("Invesco India Growth Opp",         0.158, 0.17, 1.95),
    ("L&T Midcap Fund",                  0.172, 0.19, 1.90),
    ("Edelweiss Large Cap Fund",         0.123, 0.13, 2.00),
    ("PGIM India Midcap Opp",            0.180, 0.19, 0.41),
    ("Quant Active Fund",                0.215, 0.23, 0.58),
    ("Motilal Oswal Midcap 30",          0.183, 0.20, 1.33),
    ("Sundaram Mid Cap Fund",            0.166, 0.18, 1.68),
    ("IDFC Sterling Value Fund",         0.152, 0.17, 1.72),
    ("Navi Large Cap Equity",            0.125, 0.13, 0.10),
    ("Zerodha Large Cap Fund",           0.129, 0.14, 0.11),
    ("JM Flexicap Fund",                 0.144, 0.16, 0.40),
    ("Bandhan Core Equity",              0.137, 0.15, 1.55),
    ("Baroda BNP Paribas Mid Cap",       0.170, 0.18, 1.64),
    ("Mahindra Manulife Multi Cap",      0.160, 0.17, 0.50),
    ("WhiteOak Capital Mid Cap",         0.178, 0.19, 0.55),
    ("360 One Focused Equity",           0.168, 0.18, 1.10),
    ("Samco Flexi Cap Fund",             0.145, 0.16, 0.92),
    ("Groww Large Cap Fund",             0.126, 0.13, 0.12),
    ("Groww Nifty 50 ETF FoF",           0.119, 0.13, 0.05),
    ("Axis Mid Cap Fund",                0.173, 0.18, 1.71),
    ("Kotak Bluechip Fund",              0.131, 0.13, 1.63),
    ("Tata Mid Cap Growth Fund",         0.176, 0.19, 1.80),
    ("HDFC Small Cap Fund",              0.196, 0.21, 1.79),
    ("SBI Bluechip Fund",                0.127, 0.13, 1.20),
    ("ICICI Pru Smallcap Fund",          0.198, 0.22, 1.90),
]

dates = pd.bdate_range(end='2026-06-27', periods=1304)
N_DAYS = len(dates)

# ── Market benchmarks ──────────────────────────────────────────────────
mkt_annual = 0.136; mkt_vol = 0.145
mkt_daily  = np.random.normal(mkt_annual/252, mkt_vol/np.sqrt(252), N_DAYS); mkt_daily[0]=0
nifty100_nav = 100 * np.cumprod(1 + mkt_daily)
nifty100_ret = pd.Series(mkt_daily, index=dates)
n50_daily    = mkt_daily * 0.97 + np.random.normal(0, 0.003, N_DAYS)
nifty50_nav  = 100 * np.cumprod(1 + n50_daily)
nifty50_ret  = pd.Series(n50_daily, index=dates)

# ── Fund NAV history ───────────────────────────────────────────────────
scheme_records, nav_records = [], []
for idx, (name, ann_ret, ann_vol, exp_ratio) in enumerate(FUNDS, start=1):
    beta_true  = np.random.uniform(0.7, 1.3)
    alpha_true = (ann_ret - beta_true * mkt_annual) / 252
    idio       = np.random.normal(0, ann_vol/np.sqrt(252)*0.5, N_DAYS)
    returns    = alpha_true + beta_true * mkt_daily + idio; returns[0]=0
    nav_start  = np.random.uniform(15, 200)
    nav_values = nav_start * np.cumprod(1 + returns)
    scheme_records.append({'scheme_id':idx,'scheme_name':name,'expense_ratio':exp_ratio})
    for d, nav in zip(dates, nav_values):
        nav_records.append({'scheme_id':idx,'scheme_name':name,'nav_date':d,'nav_value':nav})

nav_df    = pd.DataFrame(nav_records)
scheme_df = pd.DataFrame(scheme_records)
nav_pivot = nav_df.pivot_table(index='nav_date', columns='scheme_id', values='nav_value')
daily_returns = nav_pivot.pct_change()

print(f'✓ {nav_df.shape[0]:,} NAV records | {nav_pivot.shape[1]} schemes | {nav_pivot.shape[0]} trading days')
print(f'  Date range: {dates[0].date()} → {dates[-1].date()}')
nav_df.head(3)


## Task 1 · Daily Returns
Formula: `daily_return = NAV_t / NAV_t-1 − 1`

We validate distribution looks reasonable: mild positive skew, leptokurtic (fat tails) — typical for equity mutual funds.

In [ ]:
all_ret = daily_returns.values.flatten()
all_ret = all_ret[~np.isnan(all_ret)]

print('Daily Return Statistics (all 40 schemes pooled)')
print(f'  Mean       : {all_ret.mean()*100:.4f}%')
print(f'  Std Dev    : {all_ret.std()*100:.4f}%')
print(f'  Skewness   : {skew(all_ret):.4f}')
print(f'  Kurtosis   : {kurtosis(all_ret):.4f}')
print(f'  Min / Max  : {all_ret.min()*100:.2f}% / {all_ret.max()*100:.2f}%')

fig, axes = plt.subplots(2, 2, figsize=(14,10))
fig.suptitle('Task 1 — Daily Returns Validation', fontsize=14, fontweight='bold')

ax = axes[0,0]
ax.hist(all_ret*100, bins=120, edgecolor='none', alpha=0.75, color='steelblue')
ax.axvline(all_ret.mean()*100, color='red', ls='--', lw=1.5, label=f'Mean={all_ret.mean()*100:.3f}%')
ax.axvline(np.median(all_ret)*100, color='green', ls='--', lw=1.5, label=f'Median={np.median(all_ret)*100:.3f}%')
ax.set_xlabel('Daily Return (%)'); ax.set_ylabel('Frequency')
ax.set_title('Distribution — All 40 Schemes'); ax.legend()

ax = axes[0,1]
for col in daily_returns.columns[:6]:
    ax.plot(daily_returns.index, daily_returns[col]*100, alpha=0.55, lw=0.6)
ax.set_xlabel('Date'); ax.set_ylabel('Daily Return (%)'); ax.set_title('Time Series — 6 Sample Funds')

ax = axes[1,0]
data_bx = [daily_returns[c].dropna()*100 for c in daily_returns.columns[:15]]
bp = ax.boxplot(data_bx, patch_artist=True)
for p in bp['boxes']: p.set_facecolor('steelblue'); p.set_alpha(0.6)
ax.set_ylabel('Daily Return (%)'); ax.set_title('Box Plot — First 15 Schemes')
ax.set_xticklabels([f'F{i+1}' for i in range(15)], fontsize=7)

ax = axes[1,1]
sk = daily_returns.skew(); ku = daily_returns.kurtosis()
sc = ax.scatter(sk, ku, alpha=0.7, s=80, c=ku, cmap='RdYlGn_r', edgecolors='black', lw=0.4)
ax.axhline(0, color='red', ls='--', alpha=0.5); ax.axvline(0, color='red', ls='--', alpha=0.5)
ax.set_xlabel('Skewness'); ax.set_ylabel('Excess Kurtosis'); ax.set_title('Return Shape — All 40')
plt.colorbar(sc, ax=ax, label='Kurt')

plt.tight_layout()
plt.savefig(f'{OUTPUT}/01_daily_returns_validation.png', dpi=150, bbox_inches='tight')
plt.show(); print('✓ Saved: 01_daily_returns_validation.png')


## Task 2 · CAGR — 1yr / 3yr / 5yr
Formula: `CAGR = (NAV_end / NAV_start)^(1/n) − 1`

In [ ]:
def cagr(series, years):
    s = series.dropna()
    if len(s)<2 or s.iloc[0]<=0: return np.nan
    return ((s.iloc[-1]/s.iloc[0])**(1/years) - 1)*100

cagr_rows = []
for sid in nav_pivot.columns:
    nav  = nav_pivot[sid]
    name = scheme_df.loc[scheme_df.scheme_id==sid,'scheme_name'].values[0]
    cagr_rows.append({'scheme_id':sid,'scheme_name':name,
        'cagr_1yr':cagr(nav.tail(252),1), 'cagr_3yr':cagr(nav.tail(756),3),
        'cagr_5yr':cagr(nav.tail(1260),5), 'nav_start':nav.dropna().iloc[0], 'nav_end':nav.dropna().iloc[-1]})
cagr_df = pd.DataFrame(cagr_rows).sort_values('cagr_3yr',ascending=False).reset_index(drop=True)
print('CAGR Comparison Table — All 40 Schemes')
cagr_df[['scheme_name','cagr_1yr','cagr_3yr','cagr_5yr']].round(2)


In [ ]:
fig, axes = plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Task 2 — CAGR Analysis', fontsize=14, fontweight='bold')
top10 = cagr_df.head(10); x=np.arange(len(top10)); w=0.22
ax=axes[0,0]
ax.bar(x-w,top10.cagr_1yr,w,label='1Y',alpha=0.85)
ax.bar(x,top10.cagr_3yr,w,label='3Y',alpha=0.85)
ax.bar(x+w,top10.cagr_5yr,w,label='5Y',alpha=0.85)
ax.set_ylabel('CAGR (%)'); ax.set_title('CAGR — Top 10 Schemes')
ax.set_xticks(x); ax.set_xticklabels([s[:12] for s in top10.scheme_name],rotation=40,ha='right',fontsize=7)
ax.legend()
ax=axes[0,1]
valid=cagr_df.dropna(subset=['cagr_3yr','cagr_5yr'])
ax.scatter(valid.cagr_3yr,valid.cagr_5yr,alpha=0.7,s=80,edgecolors='black',lw=0.4)
ax.set_xlabel('3Y CAGR (%)'); ax.set_ylabel('5Y CAGR (%)'); ax.set_title('3Y vs 5Y CAGR')
ax=axes[1,0]
bp=ax.boxplot([cagr_df.cagr_1yr.dropna(),cagr_df.cagr_3yr.dropna(),cagr_df.cagr_5yr.dropna()],labels=['1Y','3Y','5Y'],patch_artist=True)
for p,c in zip(bp['boxes'],['#4e79a7','#f28e2b','#59a14f']): p.set_facecolor(c); p.set_alpha(0.7)
ax.set_ylabel('CAGR (%)'); ax.set_title('CAGR Distribution')
ax=axes[1,1]; ax.axis('off')
txt=(f"CAGR Summary\n{'─'*30}\n"
     f"{'Period':<10}{'Mean':>8}{'Median':>8}{'Max':>8}\n"
     f"{'1 Year':<10}{cagr_df.cagr_1yr.mean():>7.2f}%{cagr_df.cagr_1yr.median():>7.2f}%{cagr_df.cagr_1yr.max():>7.2f}%\n"
     f"{'3 Year':<10}{cagr_df.cagr_3yr.mean():>7.2f}%{cagr_df.cagr_3yr.median():>7.2f}%{cagr_df.cagr_3yr.max():>7.2f}%\n"
     f"{'5 Year':<10}{cagr_df.cagr_5yr.mean():>7.2f}%{cagr_df.cagr_5yr.median():>7.2f}%{cagr_df.cagr_5yr.max():>7.2f}%\n")
ax.text(0.05,0.55,txt,fontsize=11,family='monospace',va='center',bbox=dict(boxstyle='round',facecolor='#f0f4f8',alpha=0.8))
plt.tight_layout()
plt.savefig(f'{OUTPUT}/02_cagr_analysis.png',dpi=150,bbox_inches='tight')
plt.show(); print('✓ Saved: 02_cagr_analysis.png')


## Task 3 · Sharpe Ratio
Formula: `Sharpe = (Rp − Rf) / σp × √252` | Rf = 6.5% (RBI repo rate proxy)

## Task 4 · Sortino Ratio
Same as Sharpe but σ = standard deviation of **negative** daily returns only.

In [ ]:
Rf=0.065; rf_daily=Rf/252; LOOKBACK=756

def sharpe(ret, rf=rf_daily):
    r=ret.dropna()
    if len(r)<30: return np.nan
    ex=r.mean()-rf; v=r.std()
    return (ex/v)*np.sqrt(252) if v else np.nan

def sortino(ret, rf=rf_daily):
    r=ret.dropna()
    if len(r)<30: return np.nan
    ex=r.mean()-rf; down=r[r<0]; dv=down.std()
    return (ex/dv)*np.sqrt(252) if (len(down)>0 and dv) else np.nan

risk_rows=[]
for sid in daily_returns.columns:
    r3=daily_returns[sid].tail(LOOKBACK)
    name=scheme_df.loc[scheme_df.scheme_id==sid,'scheme_name'].values[0]
    risk_rows.append({'scheme_id':sid,'scheme_name':name,
        'sharpe_ratio':sharpe(r3),'sortino_ratio':sortino(r3),
        'volatility_pct':r3.std()*np.sqrt(252)*100,'ann_return_pct':r3.mean()*252*100})
risk_df=pd.DataFrame(risk_rows).sort_values('sharpe_ratio',ascending=False).reset_index(drop=True)
print('Sharpe & Sortino Rankings — All 40 Funds')
risk_df[['scheme_name','sharpe_ratio','sortino_ratio','volatility_pct','ann_return_pct']].round(3)


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Task 3&4 — Sharpe & Sortino Ratios (Rf=6.5%)',fontsize=14,fontweight='bold')
ax=axes[0,0]; top_s=risk_df.head(15)
colors=['#2ca02c' if x>1 else '#ff7f0e' if x>0.5 else '#d62728' for x in top_s.sharpe_ratio]
ax.barh(range(len(top_s)),top_s.sharpe_ratio,color=colors,alpha=0.8,edgecolor='white')
ax.set_yticks(range(len(top_s))); ax.set_yticklabels([s[:28] for s in top_s.scheme_name],fontsize=7)
ax.set_xlabel('Sharpe Ratio'); ax.set_title('Sharpe Rankings — Top 15')
ax.axvline(1.0,color='black',ls='--',lw=1,alpha=0.5)
patches=[mpatches.Patch(color='#2ca02c',label='>1 Good'),mpatches.Patch(color='#ff7f0e',label='0.5–1 Ok'),mpatches.Patch(color='#d62728',label='<0.5 Poor')]
ax.legend(handles=patches,fontsize=8)
ax=axes[0,1]; v=risk_df.dropna(subset=['sharpe_ratio','sortino_ratio'])
sc=ax.scatter(v.sharpe_ratio,v.sortino_ratio,alpha=0.75,s=80,c=v.ann_return_pct,cmap='RdYlGn',edgecolors='black',lw=0.3)
plt.colorbar(sc,ax=ax,label='Ann Return (%)')
ax.set_xlabel('Sharpe Ratio'); ax.set_ylabel('Sortino Ratio'); ax.set_title('Sharpe vs Sortino')
ax=axes[1,0]
ax.scatter(risk_df.volatility_pct,risk_df.ann_return_pct,alpha=0.75,s=80,edgecolors='black',lw=0.3)
for _,row in risk_df.head(5).iterrows(): ax.annotate(row.scheme_name[:12],(row.volatility_pct,row.ann_return_pct),fontsize=7)
ax.set_xlabel('Volatility (%)'); ax.set_ylabel('Ann Return (%)'); ax.set_title('Risk–Return Profile')
ax=axes[1,1]
ax.hist(risk_df.sharpe_ratio.dropna(),bins=15,edgecolor='black',alpha=0.75)
ax.axvline(risk_df.sharpe_ratio.mean(),color='red',ls='--',lw=1.5,label=f'Mean={risk_df.sharpe_ratio.mean():.2f}')
ax.set_xlabel('Sharpe Ratio'); ax.set_ylabel('Frequency'); ax.set_title('Distribution of Sharpe Ratios'); ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT}/03_sharpe_sortino.png',dpi=150,bbox_inches='tight')
plt.show(); print('✓ Saved: 03_sharpe_sortino.png')


## Task 5 · Alpha & Beta (OLS vs Nifty 100)
Formula: `fund_return = α + β × nifty100_return + ε`
- **Beta** = slope (market sensitivity)
- **Alpha** = intercept × 252 (annualised excess return vs market)
- Using `scipy.stats.linregress`

In [ ]:
ab_rows=[]
for sid in daily_returns.columns:
    fund_ret=daily_returns[sid].dropna()
    common=fund_ret.index.intersection(nifty100_ret.index)
    if len(common)<60: continue
    fr=fund_ret[common].values; nr=nifty100_ret[common].values
    slope,intercept,r_val,p_val,se=linregress(nr,fr)
    name=scheme_df.loc[scheme_df.scheme_id==sid,'scheme_name'].values[0]
    ab_rows.append({'scheme_id':sid,'scheme_name':name,
        'alpha':intercept*252*100,'beta':slope,'r_squared':r_val**2,'p_value':p_val,'observations':len(common)})
alpha_beta_df=pd.DataFrame(ab_rows).sort_values('alpha',ascending=False).reset_index(drop=True)
alpha_beta_df.insert(0,'alpha_rank',range(1,len(alpha_beta_df)+1))
print('Alpha & Beta — All 40 Funds (ranked by Alpha)')
alpha_beta_df[['alpha_rank','scheme_name','alpha','beta','r_squared','p_value']].round(4)


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Task 5 — Alpha & Beta (OLS vs Nifty 100)',fontsize=14,fontweight='bold')
ax=axes[0,0]; top_a=alpha_beta_df.head(15)
colors=['#2ca02c' if x>0 else '#d62728' for x in top_a.alpha]
ax.barh(range(len(top_a)),top_a.alpha,color=colors,alpha=0.8,edgecolor='white')
ax.set_yticks(range(len(top_a))); ax.set_yticklabels([s[:28] for s in top_a.scheme_name],fontsize=7)
ax.set_xlabel('Alpha (% p.a.)'); ax.set_title('Alpha Rankings — Top 15'); ax.axvline(0,color='black',lw=1)
ax=axes[0,1]
sc=ax.scatter(alpha_beta_df.beta,alpha_beta_df.alpha,s=80,alpha=0.75,c=alpha_beta_df.r_squared,cmap='RdYlGn',edgecolors='black',lw=0.3)
ax.axhline(0,color='red',ls='--',alpha=0.5); ax.axvline(1,color='blue',ls='--',alpha=0.5,label='β=1')
plt.colorbar(sc,ax=ax,label='R²'); ax.set_xlabel('Beta'); ax.set_ylabel('Alpha (% p.a.)'); ax.set_title('Alpha vs Beta'); ax.legend()
ax=axes[1,0]
ax.hist(alpha_beta_df.beta,bins=15,edgecolor='black',alpha=0.75)
ax.axvline(1.0,color='red',ls='--',lw=2,label='β=1 (market)')
ax.set_xlabel('Beta'); ax.set_ylabel('Frequency'); ax.set_title('Distribution of Beta'); ax.legend()
ax=axes[1,1]
ax.hist(alpha_beta_df.r_squared,bins=15,edgecolor='black',alpha=0.75,color='teal')
ax.set_xlabel('R²'); ax.set_ylabel('Frequency'); ax.set_title('Distribution of R²')
plt.tight_layout()
plt.savefig(f'{OUTPUT}/04_alpha_beta.png',dpi=150,bbox_inches='tight')
plt.show(); print('✓ Saved: 04_alpha_beta.png')
# ── Export alpha_beta.csv ────────────────────────────────────────────────
alpha_beta_df.round(6).to_csv(f'{OUTPUT}/alpha_beta.csv',index=False)
print(f'✓ Exported alpha_beta.csv ({len(alpha_beta_df)} funds)')


## Task 6 · Maximum Drawdown
Formula: `MaxDD = min(NAV_t / running_max_t − 1)`

Peak → Trough → Recovery dates are also captured.

In [ ]:
def max_drawdown_stats(nav_series):
    nav=nav_series.dropna()
    if len(nav)<2: return np.nan, pd.NaT, pd.NaT, pd.NaT
    running_max=nav.cummax()
    dd=(nav/running_max)-1
    trough_idx=dd.idxmin(); max_dd=dd[trough_idx]*100
    peak_nav=running_max[trough_idx]
    peak_idx=nav[:trough_idx][nav[:trough_idx]>=peak_nav].index[-1] if (nav[:trough_idx]>=peak_nav).any() else nav.index[0]
    post=nav[trough_idx:]; rec_mask=post>=peak_nav
    recovery_idx=post[rec_mask].index[0] if rec_mask.any() else pd.NaT
    return max_dd, peak_idx, trough_idx, recovery_idx

dd_rows=[]
for sid in nav_pivot.columns:
    nav=nav_pivot[sid]
    name=scheme_df.loc[scheme_df.scheme_id==sid,'scheme_name'].values[0]
    max_dd,peak,trough,recovery=max_drawdown_stats(nav)
    rec_days=(recovery-peak).days if pd.notna(recovery) and pd.notna(peak) else np.nan
    dd_rows.append({'scheme_id':sid,'scheme_name':name,'max_drawdown':max_dd,
        'peak_date':peak,'trough_date':trough,'recovery_date':recovery,'recovery_days':rec_days})
drawdown_df=pd.DataFrame(dd_rows).sort_values('max_drawdown').reset_index(drop=True)
print('Maximum Drawdown — All 40 Funds')
drawdown_df[['scheme_name','max_drawdown','peak_date','trough_date','recovery_date','recovery_days']].round(2)


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Task 6 — Maximum Drawdown Analysis',fontsize=14,fontweight='bold')
worst10=drawdown_df.head(10)
ax=axes[0,0]
ax.barh(range(len(worst10)),worst10.max_drawdown,color='darkred',alpha=0.75,edgecolor='white')
ax.set_yticks(range(len(worst10))); ax.set_yticklabels([s[:28] for s in worst10.scheme_name],fontsize=7)
ax.set_xlabel('Max Drawdown (%)'); ax.set_title('10 Worst Drawdowns')
ax=axes[0,1]
ax.hist(drawdown_df.max_drawdown.dropna(),bins=15,edgecolor='black',alpha=0.75,color='tomato')
ax.axvline(drawdown_df.max_drawdown.mean(),color='black',ls='--',label=f'Mean={drawdown_df.max_drawdown.mean():.1f}%')
ax.set_xlabel('Max Drawdown (%)'); ax.set_ylabel('Frequency'); ax.set_title('Distribution of Max DD'); ax.legend()
ax=axes[1,0]
ax.hist(drawdown_df.recovery_days.dropna(),bins=15,edgecolor='black',alpha=0.75)
ax.set_xlabel('Recovery Days'); ax.set_ylabel('Frequency'); ax.set_title('Recovery Period Distribution')
ax=axes[1,1]; valid_dd=drawdown_df.dropna(subset=['max_drawdown','recovery_days'])
ax.scatter(abs(valid_dd.max_drawdown),valid_dd.recovery_days,alpha=0.7,s=80,c='darkred',edgecolors='black',lw=0.3)
ax.set_xlabel('Max Drawdown Magnitude (%)'); ax.set_ylabel('Recovery Days'); ax.set_title('Drawdown vs Recovery Time')
plt.tight_layout()
plt.savefig(f'{OUTPUT}/05_drawdown.png',dpi=150,bbox_inches='tight')
plt.show(); print('✓ Saved: 05_drawdown.png')


## Task 7 · Fund Scorecard (0–100)
Composite score using percentile ranks:

| Weight | Metric | Direction |
|--------|--------|-----------|
| 30% | 3Y CAGR | Higher = better |
| 25% | Sharpe Ratio | Higher = better |
| 20% | Alpha | Higher = better |
| 15% | Expense Ratio | **Lower** = better |
| 10% | Max Drawdown | **Less negative** = better |

In [ ]:
sc=cagr_df[['scheme_id','scheme_name','cagr_1yr','cagr_3yr','cagr_5yr']].copy()
sc=sc.merge(risk_df[['scheme_id','sharpe_ratio','sortino_ratio','volatility_pct']],on='scheme_id',how='left')
sc=sc.merge(alpha_beta_df[['scheme_id','alpha','beta','r_squared']],on='scheme_id',how='left')
sc=sc.merge(scheme_df[['scheme_id','expense_ratio']],on='scheme_id',how='left')
sc=sc.merge(drawdown_df[['scheme_id','max_drawdown']],on='scheme_id',how='left')

def pct_rank(s, higher_is_better=True):
    r=s.rank(pct=True,method='average')*100
    return r if higher_is_better else 100-r

sc['r_cagr3']   =pct_rank(sc.cagr_3yr,True)
sc['r_sharpe']  =pct_rank(sc.sharpe_ratio,True)
sc['r_alpha']   =pct_rank(sc.alpha,True)
sc['r_expense'] =pct_rank(sc.expense_ratio,False)
sc['r_drawdown']=pct_rank(sc.max_drawdown,False)

sc['composite_score']=(0.30*sc.r_cagr3.fillna(50)+0.25*sc.r_sharpe.fillna(50)+
                       0.20*sc.r_alpha.fillna(50)+0.15*sc.r_expense.fillna(50)+
                       0.10*sc.r_drawdown.fillna(50))

scorecard_final=sc.sort_values('composite_score',ascending=False).reset_index(drop=True)
scorecard_final.insert(0,'rank',range(1,len(scorecard_final)+1))

score_export=scorecard_final[['rank','scheme_id','scheme_name','composite_score',
    'cagr_1yr','cagr_3yr','cagr_5yr','sharpe_ratio','sortino_ratio',
    'alpha','beta','r_squared','expense_ratio','max_drawdown','volatility_pct']].round(4)
score_export.to_csv(f'{OUTPUT}/fund_scorecard.csv',index=False)
print(f'✓ Exported fund_scorecard.csv ({len(score_export)} funds)')
print('\nTop 10 Funds:')
scorecard_final[['rank','scheme_name','composite_score','cagr_3yr','sharpe_ratio','alpha']].head(10).round(2)


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Task 7 — Fund Scorecard (0–100 Composite)',fontsize=14,fontweight='bold')
top15=scorecard_final.head(15)
ax=axes[0,0]
bar_colors=plt.cm.RdYlGn(top15.composite_score/100)
ax.barh(range(len(top15)),top15.composite_score,color=bar_colors,edgecolor='white')
ax.set_yticks(range(len(top15))); ax.set_yticklabels([s[:28] for s in top15.scheme_name],fontsize=7)
ax.set_xlim(0,100); ax.set_xlabel('Score'); ax.set_title('Top 15 Funds — Composite Scorecard')
ax=axes[0,1]
cats=['3Y CAGR\n(30%)','Sharpe\n(25%)','Alpha\n(20%)','Expense\n(15%)','Drawdown\n(10%)']
ax.pie([0.30,0.25,0.20,0.15,0.10],labels=cats,autopct='%1.0f%%',startangle=90,
       colors=['#4e79a7','#f28e2b','#59a14f','#e15759','#76b7b2'])
ax.set_title('Score Weighting')
ax=axes[1,0]
ax.hist(scorecard_final.composite_score.dropna(),bins=20,edgecolor='black',alpha=0.75)
ax.axvline(scorecard_final.composite_score.mean(),color='red',ls='--',lw=2,
           label=f'Mean={scorecard_final.composite_score.mean():.1f}')
ax.set_xlabel('Score'); ax.set_ylabel('Frequency'); ax.set_title('Score Distribution'); ax.legend()
ax=axes[1,1]
ax.scatter(scorecard_final.cagr_3yr,scorecard_final.composite_score,alpha=0.75,s=80,edgecolors='black',lw=0.3)
for _,row in scorecard_final.head(5).iterrows(): ax.annotate(row.scheme_name[:12],(row.cagr_3yr,row.composite_score),fontsize=7)
ax.set_xlabel('3Y CAGR (%)'); ax.set_ylabel('Composite Score'); ax.set_title('Score vs 3Y CAGR')
plt.tight_layout()
plt.savefig(f'{OUTPUT}/06_fund_scorecard.png',dpi=150,bbox_inches='tight')
plt.show(); print('✓ Saved: 06_fund_scorecard.png')


## Task 8 · Benchmark Comparison — Top 5 vs Nifty 50 & Nifty 100
- Cumulative return chart over 3 years
- **Tracking Error** = `std(fund_return − benchmark_return) × √252`
- **Information Ratio** = `mean(active_return) / std(active_return) × √252`

In [ ]:
top5_ids=scorecard_final.head(5).scheme_id.values
LOOKBACK_BM=756; ref_dates=daily_returns.index[-LOOKBACK_BM:]
n100_cum=(1+nifty100_ret[ref_dates]).cumprod()
n50_cum =(1+nifty50_ret[ref_dates]).cumprod()

te_rows=[]
for sid in top5_ids:
    fr=daily_returns[sid][ref_dates]; nr=nifty100_ret[ref_dates]
    name=scorecard_final.loc[scorecard_final.scheme_id==sid,'scheme_name'].values[0]
    td=fr-nr
    te=td.std()*np.sqrt(252)*100
    ir=(td.mean()/td.std())*np.sqrt(252) if td.std() else np.nan
    te_rows.append({'scheme_name':name,'tracking_error_%':te,'information_ratio':ir,'outperformance_%':td.sum()*252*100})
te_df=pd.DataFrame(te_rows)
print('Tracking Error — Top 5 Funds vs Nifty 100')
te_df.round(3)


In [ ]:
fig,axes=plt.subplots(2,1,figsize=(14,11))
fig.suptitle('Task 8 — Top 5 Funds vs Nifty 50 & Nifty 100 (3-Year)',fontsize=14,fontweight='bold')

ax=axes[0]
ax.plot(ref_dates,n50_cum.values, label='Nifty 50', lw=2.2,color='black',ls='--')
ax.plot(ref_dates,n100_cum.values,label='Nifty 100',lw=2.2,color='dimgrey',ls=':')
fund_colors=['#e41a1c','#377eb8','#4daf4a','#ff7f00','#984ea3']
for i,sid in enumerate(top5_ids):
    fr=daily_returns[sid][ref_dates]; cum=(1+fr).cumprod()
    name=scorecard_final.loc[scorecard_final.scheme_id==sid,'scheme_name'].values[0]
    ax.plot(ref_dates,cum.values,label=name[:30],lw=1.8,color=fund_colors[i])
ax.set_xlabel('Date'); ax.set_ylabel('Cumulative Return (base=1)')
ax.set_title('Cumulative Performance: Top 5 Funds vs Benchmarks'); ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

ax=axes[1]; x=np.arange(len(te_df)); w=0.35
ax.bar(x-w/2,te_df['tracking_error_%'],w,label='Tracking Error (%)',alpha=0.8,color='steelblue',edgecolor='white')
ax2=ax.twinx()
ax2.bar(x+w/2,te_df['information_ratio'],w,label='Information Ratio',alpha=0.8,color='coral',edgecolor='white')
ax.set_ylabel('Tracking Error (% p.a.)',color='steelblue'); ax2.set_ylabel('Information Ratio',color='coral')
ax.set_title('Tracking Error & Information Ratio — Top 5 vs Nifty 100')
ax.set_xticks(x); ax.set_xticklabels([s[:22] for s in te_df.scheme_name],rotation=25,ha='right',fontsize=8)
lines=[mpatches.Patch(color='steelblue',label='Tracking Error (%)'),mpatches.Patch(color='coral',label='Information Ratio')]
ax.legend(handles=lines,loc='upper right')
plt.tight_layout()
plt.savefig(f'{OUTPUT}/07_benchmark_comparison.png',dpi=150,bbox_inches='tight')
plt.show(); print('✓ Saved: 07_benchmark_comparison.png  ← DELIVERABLE')


## Summary & Deliverables

In [ ]:
print('='*65)
print('ALL TASKS COMPLETE ✓')
print('='*65)
print(f'\nTop 5 Funds by Composite Score:')
for _,row in scorecard_final.head(5).iterrows():
    print(f'  #{int(row.rank)} {row.scheme_name:<40}  Score={row.composite_score:.1f}')
print(f'\nAggregate Statistics:')
print(f'  Avg 3Y CAGR     : {cagr_df.cagr_3yr.mean():.2f}%')
print(f'  Avg Sharpe      : {risk_df.sharpe_ratio.mean():.3f}')
print(f'  Avg Sortino     : {risk_df.sortino_ratio.mean():.3f}')
print(f'  Avg Alpha       : {alpha_beta_df.alpha.mean():.2f}% p.a.')
print(f'  Avg Beta        : {alpha_beta_df.beta.mean():.3f}')
print(f'  Avg Max Drawdown: {drawdown_df.max_drawdown.mean():.2f}%')
print(f'\nDeliverables written to: {OUTPUT}/')
for f in sorted(os.listdir(OUTPUT)): print(f'  ✓ {f}')
